# 🔁 Refund Decision Simulator

## Comparing Rule-Based vs ML Approaches for Economic Optimization

---

### 📋 Problem Statement

Online platforms must make thousands of refund decisions daily, balancing:
- **Fraud risk** — approving fraudulent refunds incurs direct financial loss
- **Customer retention** — wrongly denying legitimate refunds drives churn
- **Operational cost** — every approved refund has a direct monetary cost

This project demonstrates that **optimizing for classification accuracy alone does not guarantee optimal economic outcomes**, and compares rule-based heuristics against multiple ML models using a custom economic cost function.

### 🎯 Objectives
1. Generate synthetic refund decision data
2. Implement 3 rule-based strategies (simple, conservative, lenient)
3. Train 4 ML models with hyperparameter tuning
4. Evaluate all approaches on the **same test set** (fair comparison)
5. Compare using both accuracy AND economic cost
6. Determine which approach minimizes total business cost

---
## 1️⃣ Setup & Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project modules
from src.config import Config
from src.data_generator import generate_dataset
from src.rule_engine import RuleEngine
from src.model import ModelPipeline, get_available_models
from src.metrics import EconomicMetrics, classification_metrics, get_classification_report
from src.visualization import (
    plot_confusion_matrix,
    plot_cost_comparison,
    plot_feature_importance,
    plot_model_comparison,
    plot_roc_curves,
    plot_data_distribution,
)

# Initialize configuration
config = Config(random_seed=42, n_samples=1000)
print(f"Random Seed: {config.random_seed}")
print(f"Sample Size: {config.n_samples}")
print(f"Test Size: {config.test_size}")
print(f"Available Models: {get_available_models()}")

---
## 2️⃣ Data Generation & Exploratory Analysis

In [ ]:
# Generate synthetic dataset
data = generate_dataset(config=config)

print(f"Dataset Shape: {data.shape}")
print(f"\nClass Distribution:")
print(data['refunded'].value_counts().to_string())
print(f"\nRefund Rate: {data['refunded'].mean():.1%}")
print(f"\n{'='*50}")
print("\nStatistical Summary:")
data.describe().round(2)

In [ ]:
# Visualize feature distributions
fig = plot_data_distribution(data)
plt.show()

In [ ]:
# Correlation matrix
from src.visualization import apply_style
apply_style()

fig, ax = plt.subplots(figsize=(8, 6))
corr = data.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, linecolor='#30363d')
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 3️⃣ Rule-Based Decision Systems

We implement three heuristic strategies with different risk tolerances:

| Strategy | Bias | Description |
|----------|------|-------------|
| **Simple** | Balanced | Delay > 30 → approve, past refunds > 3 → reject |
| **Conservative** | Toward rejection | Fraud > 0.5 → reject, stricter thresholds |
| **Lenient** | Toward approval | Only fraud > 0.8 → reject, lower thresholds |

In [ ]:
# Apply all rule-based strategies
engine = RuleEngine()
rule_predictions = engine.predict_all_strategies(data)

print("Rule-Based Strategy Results:")
print("=" * 50)
for strategy_name, preds in rule_predictions.items():
    metrics = classification_metrics(data['refunded'].values, preds)
    print(f"\n📌 {strategy_name.upper()}")
    print(f"   Accuracy:  {metrics['accuracy']:.3f}")
    print(f"   Precision: {metrics['precision']:.3f}")
    print(f"   Recall:    {metrics['recall']:.3f}")
    print(f"   F1 Score:  {metrics['f1_score']:.3f}")
    print(f"   Approval Rate: {preds.mean():.1%}")

---
## 4️⃣ Machine Learning Models

We train four models, each with:
- **StandardScaler** preprocessing
- **5-fold cross-validation**
- **GridSearchCV** hyperparameter tuning

In [ ]:
# Prepare data with train/test split
pipeline = ModelPipeline(config=config)
X_train, X_test, y_train, y_test = pipeline.prepare_data(data)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {list(X_train.columns)}")
print(f"\nTrain refund rate: {y_train.mean():.1%}")
print(f"Test refund rate:  {y_test.mean():.1%}")

In [ ]:
# Train all models with hyperparameter tuning
print("Training models with GridSearchCV...\n")
pipeline.train_all(tune_hyperparams=True)

# Display results summary
results = pipeline.get_results_summary()
print("\n📊 Model Performance Summary:")
print("=" * 70)
print(results[['Model', 'CV_Mean_Accuracy', 'CV_Std', 'Test_Accuracy']].to_string(index=False))

# Show best hyperparameters
print("\n🔧 Best Hyperparameters:")
print("=" * 70)
for name, params in pipeline.best_params.items():
    print(f"\n{name}:")
    for k, v in params.items():
        print(f"  {k.replace('classifier__', '')}: {v}")

In [ ]:
# Visualize model comparison
fig = plot_model_comparison(results)
plt.show()

---
## 5️⃣ Detailed Classification Reports

In [ ]:
# Print detailed reports for each model
for model_name in pipeline.models:
    preds = pipeline.predict(model_name)
    print(f"\n{'='*50}")
    print(f"📋 {model_name} - Classification Report")
    print("=" * 50)
    report = get_classification_report(y_test.values, preds)
    print(report.round(3).to_string())

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
model_names = list(pipeline.models.keys())

for i, (ax, name) in enumerate(zip(axes.flatten(), model_names)):
    preds = pipeline.predict(name)
    from src.metrics import get_confusion_matrix
    cm = get_confusion_matrix(y_test.values, preds)
    from src.visualization import apply_style
    apply_style()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Rejected', 'Refunded'],
                yticklabels=['Rejected', 'Refunded'],
                ax=ax, linewidths=0.5, linecolor='#30363d')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

# Hide unused axes if less than 4 models
for j in range(len(model_names), 4):
    axes.flatten()[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6️⃣ ROC Curves

In [ ]:
# Generate ROC curves for all models
probas_dict = {}
for name in pipeline.models:
    probas_dict[name] = pipeline.predict_proba(name)

fig = plot_roc_curves(y_test.values, probas_dict)
plt.show()

---
## 7️⃣ Feature Importance Analysis

In [ ]:
# Feature importance for tree-based models
tree_models = [m for m in pipeline.models if m != 'LogisticRegression']

fig, axes = plt.subplots(1, len(tree_models), figsize=(6*len(tree_models), 5))
if len(tree_models) == 1:
    axes = [axes]

apply_style()
for ax, name in zip(axes, tree_models):
    importance = pipeline.get_feature_importance(name)
    sorted_imp = importance.sort_values(ascending=True)
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(sorted_imp)))
    ax.barh(sorted_imp.index, sorted_imp.values, color=colors,
            edgecolor='#30363d', linewidth=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance — Tree-Based Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Also show LogisticRegression coefficients
print("\n📊 Logistic Regression — Absolute Coefficient Magnitudes:")
lr_importance = pipeline.get_feature_importance('LogisticRegression')
print(lr_importance.round(4).to_string())

---
## 8️⃣ Economic Cost Analysis (Fair Comparison)

### Cost Model
| Scenario | Cost |
|----------|------|
| Approve refund | `order_amount` |
| Approve refund + high fraud | `order_amount × fraud_penalty_multiplier` |
| Deny legitimate refund | `retention_value` (₹500 customer churn risk) |

> ⚠️ **All strategies are evaluated on the same test set** for a fair comparison.

In [ ]:
# Prepare economic evaluation — ALL on the SAME test set
econ = EconomicMetrics(config=config)
test_data = data.loc[X_test.index].reset_index(drop=True)

# Collect all predictions for test set
all_predictions = {}

# Rule-based (re-evaluate on test set only)
for strategy_name in RuleEngine.STRATEGIES:
    rule_preds = engine.predict(test_data, strategy=strategy_name)
    all_predictions[f"Rule: {strategy_name}"] = rule_preds

# ML models (already evaluated on test set)
for model_name in pipeline.models:
    all_predictions[f"ML: {model_name}"] = pipeline.predict(model_name)

# Compare all strategies
cost_comparison = econ.compare_strategies(test_data, all_predictions)

print("💰 Economic Cost Comparison (Test Set):")
print("=" * 70)
print(cost_comparison.round(2).to_string())

In [ ]:
# Visualize cost comparison
fig = plot_cost_comparison(cost_comparison, title="Economic Cost Comparison — All Strategies")
plt.show()

In [ ]:
# Detailed cost breakdown for the best ML model
best_model = results.iloc[0]['Model']
best_preds = pipeline.predict(best_model)
breakdown = econ.cost_breakdown(test_data, best_preds)

print(f"\n🏆 Best Model: {best_model}")
print("=" * 40)
print(f"  Refund Cost:     ₹{breakdown['refund_cost']:>12,.2f}")
print(f"  Fraud Penalty:   ₹{breakdown['fraud_penalty']:>12,.2f}")
print(f"  Retention Loss:  ₹{breakdown['retention_loss']:>12,.2f}")
print(f"  ────────────────────────────────")
print(f"  TOTAL COST:      ₹{breakdown['total_cost']:>12,.2f}")

---
## 9️⃣ Accuracy vs Cost Tradeoff

In [ ]:
# Build combined accuracy + cost table
summary_rows = []

for strategy_name, preds in all_predictions.items():
    metrics = classification_metrics(test_data['refunded'].values, preds)
    cost = econ.calculate_total_cost(test_data, preds)
    summary_rows.append({
        'Strategy': strategy_name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1_score'],
        'Total Cost (₹)': cost,
        'Approval Rate': preds.mean(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Total Cost (₹)')
print("\n📊 Complete Strategy Comparison:")
print("=" * 90)
print(summary_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# Scatter plot: Accuracy vs Total Cost
apply_style()

fig, ax = plt.subplots(figsize=(10, 7))

rule_mask = summary_df['Strategy'].str.startswith('Rule')
ml_mask = summary_df['Strategy'].str.startswith('ML')

ax.scatter(summary_df.loc[rule_mask, 'Accuracy'],
           summary_df.loc[rule_mask, 'Total Cost (₹)'],
           s=150, color='#f78166', edgecolors='white', linewidth=1.5,
           label='Rule-Based', zorder=5)

ax.scatter(summary_df.loc[ml_mask, 'Accuracy'],
           summary_df.loc[ml_mask, 'Total Cost (₹)'],
           s=150, color='#58a6ff', edgecolors='white', linewidth=1.5,
           label='ML-Based', zorder=5)

# Label each point
for _, row in summary_df.iterrows():
    short_name = row['Strategy'].replace('Rule: ', 'R:').replace('ML: ', 'M:')
    ax.annotate(short_name,
                (row['Accuracy'], row['Total Cost (₹)']),
                textcoords='offset points', xytext=(10, 5),
                fontsize=8, color='#c9d1d9')

ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Economic Cost (₹)', fontsize=12, fontweight='bold')
ax.set_title('Accuracy vs Economic Cost Tradeoff', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔟 Conclusion

### Key Findings

1. **Accuracy ≠ Economic Optimality** — The model with the highest accuracy doesn't necessarily achieve the lowest economic cost. This is the central insight of cost-sensitive decision systems.

2. **Rule strategies show clear tradeoffs** — Conservative rules minimize fraud losses but increase churn costs; lenient rules maximize approvals but expose the platform to fraud.

3. **ML models provide a balanced approach** — Machine learning models, especially tree-based ones, learn nuanced decision boundaries that can balance multiple cost components simultaneously.

4. **Feature importance reveals key drivers** — Fraud score, complaint severity, and delivery delay are consistently the most important features across all models.

5. **Fair comparison is critical** — Earlier unfair comparisons (rule-based on full data vs ML on test data) can severely mislead conclusions.

### Limitations
- Synthetic dataset — real-world data would have more complex patterns
- Static cost assumptions — in practice, costs vary by customer segment
- No temporal dynamics — fraud patterns evolve over time
- Limited feature set — real systems use dozens of features

### Future Work
- Implement cost-sensitive learning (class weights, custom loss functions)
- Add threshold optimization for probability-based decisions
- Test with real-world anonymized datasets
- Build a real-time decision API